# ChatWaifu · Ayachi Nene Qwen3-TTS local fine-tuning

This notebook trains a **local research checkpoint** from game-extracted voice data. The source license declaration does not prove rights to redistribute the underlying commercial game audio. Keep the dataset, checkpoint, and generated voice private and non-commercial unless you obtain permission.

The default is an experimental 200-step 0.6B pilot using SDPA. A normal 15 GiB Colab T4 is the initial target. Listen to every Japanese and Chinese evaluation clip before opting into a full run.

In [ ]:
# Training controls
ACKNOWLEDGE_LOCAL_ONLY = True
MODEL_SIZE = "0.6B"  # @param ["0.6B", "1.7B"]
MODEL_IDS = {
    "0.6B": "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    "1.7B": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
}
MODEL_ID = MODEL_IDS[MODEL_SIZE]
PILOT_MODE = True  # @param {type:"boolean"}
EPOCHS = 2
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-6
MAX_STEPS = 200 if PILOT_MODE else 0
CODE_BATCH_SIZE = 4
ATTENTION_IMPLEMENTATION = "sdpa"
SPEAKER_NAME = "ayachi_nene_local"
QWEN_COMMIT = "022e286b98fbec7e1e916cb940cdf532cd9f488e"
assert ACKNOWLEDGE_LOCAL_ONLY, "Read RIGHTS_NOTICE.md before continuing"

In [ ]:
# Upload and safely extract the compact training bundle.
import zipfile
from pathlib import Path, PurePosixPath

from google.colab import files

uploaded = files.upload()
bundle_zip = Path(next(name for name in uploaded if name.endswith(".zip"))).resolve()
extract_root = Path("/content/chatwaifu-nene-training")
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle_zip) as archive:
    for member in archive.infolist():
        path = PurePosixPath(member.filename)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Unsafe archive member: {member.filename}")
    archive.extractall(extract_root)
roots = [path.parent for path in extract_root.glob("*/bundle_manifest.json")]
assert len(roots) == 1, f"Expected one bundle root, found {roots}"
BUNDLE_ROOT = roots[0]
print(BUNDLE_ROOT)
print((BUNDLE_ROOT / "RIGHTS_NOTICE.md").read_text())

In [ ]:
# GPU preflight: fail early on an unsuitable runtime.
import subprocess

import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3
print(f"GPU={gpu.name}, VRAM={vram_gib:.1f} GiB, bf16={torch.cuda.is_bf16_supported()}")
minimum_vram_gib = 12 if MODEL_SIZE == "0.6B" else 30
if vram_gib < minimum_vram_gib:
    raise RuntimeError(
        f"{MODEL_SIZE} training requires at least {minimum_vram_gib} GiB VRAM; "
        f"this runtime reports {vram_gib:.1f} GiB."
    )
if MODEL_SIZE == "0.6B":
    print("Experimental 0.6B path: run the pilot and listen before setting PILOT_MODE=False.")

In [ ]:
# Install a pinned Qwen source tree and its training dependencies.
import importlib
import os
import subprocess
import sys

QWEN_ROOT = Path("/content/Qwen3-TTS")
if not QWEN_ROOT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/QwenLM/Qwen3-TTS.git", str(QWEN_ROOT)], check=True
    )
subprocess.run(["git", "-C", str(QWEN_ROOT), "fetch", "origin", QWEN_COMMIT], check=True)
subprocess.run(["git", "-C", str(QWEN_ROOT), "checkout", "--detach", QWEN_COMMIT], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(QWEN_ROOT),
        "tensorboard",
        "ninja",
        "packaging",
    ],
    check=True,
)
if ATTENTION_IMPLEMENTATION == "flash_attention_2":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"],
        check=True,
    )
# Editable installs add a .pth file that the already-running Colab kernel does not reread.
if str(QWEN_ROOT) not in sys.path:
    sys.path.insert(0, str(QWEN_ROOT))
importlib.invalidate_caches()
qwen_tts = importlib.import_module("qwen_tts")

print(f"qwen_tts import OK: {qwen_tts.__file__}")
print(
    subprocess.check_output(["git", "-C", str(QWEN_ROOT), "rev-parse", "HEAD"], text=True).strip()
)

In [ ]:
# Convert selected Opus clips to the exact 24 kHz PCM16 WAV format required by Qwen.
subprocess.run(
    [
        sys.executable,
        str(BUNDLE_ROOT / "scripts/materialize_dataset.py"),
        "--bundle-root",
        str(BUNDLE_ROOT),
        "--workers",
        "4",
    ],
    check=True,
)
print((BUNDLE_ROOT / "data/materialization_report.json").read_text()[:2000])

In [ ]:
# Download the base model as a real local snapshot; the upstream saver requires local files.
from huggingface_hub import snapshot_download

model_cache = f"/content/qwen3-tts-base-{MODEL_SIZE.lower()}"
MODEL_ROOT = Path(snapshot_download(MODEL_ID, local_dir=model_cache))
assert (MODEL_ROOT / "config.json").is_file()
print(MODEL_ROOT)

In [ ]:
# Extract discrete 12 Hz audio codes with a bounded batch size.
TRAIN_RAW = BUNDLE_ROOT / "data/train_raw.jsonl"
TRAIN_CODES = BUNDLE_ROOT / "data/train_with_codes.jsonl"
subprocess.run(
    [
        sys.executable,
        str(BUNDLE_ROOT / "scripts/prepare_codes.py"),
        "--device",
        "cuda:0",
        "--input_jsonl",
        str(TRAIN_RAW),
        "--output_jsonl",
        str(TRAIN_CODES),
        "--batch_size",
        str(CODE_BATCH_SIZE),
    ],
    check=True,
)

In [ ]:
# Single-speaker SFT: 200-step pilot by default, full two epochs only by opt-in.
run_mode = "pilot" if PILOT_MODE else "full"
OUTPUT_ROOT = Path(f"/content/nene-qwen3-{MODEL_SIZE.lower()}-{run_mode}-output")
env = os.environ.copy()
env["PYTHONPATH"] = str(QWEN_ROOT / "finetuning") + os.pathsep + env.get("PYTHONPATH", "")
command = [
    sys.executable,
    str(BUNDLE_ROOT / "scripts/sft_12hz_chatwaifu.py"),
    "--init_model_path",
    str(MODEL_ROOT),
    "--output_model_path",
    str(OUTPUT_ROOT),
    "--train_jsonl",
    str(TRAIN_CODES),
    "--speaker_name",
    SPEAKER_NAME,
    "--batch_size",
    str(BATCH_SIZE),
    "--gradient_accumulation_steps",
    str(GRADIENT_ACCUMULATION_STEPS),
    "--lr",
    str(LEARNING_RATE),
    "--num_epochs",
    str(EPOCHS),
    "--max_steps",
    str(MAX_STEPS),
    "--attention_implementation",
    ATTENTION_IMPLEMENTATION,
]
subprocess.run(command, check=True, env=env)

In [ ]:
# Generate fixed Japanese/Chinese evaluation samples from the final checkpoint.
import gc
import importlib
import json
import sys

import soundfile as sf

if str(QWEN_ROOT) not in sys.path:
    sys.path.insert(0, str(QWEN_ROOT))
importlib.invalidate_caches()
Qwen3TTSModel = importlib.import_module("qwen_tts").Qwen3TTSModel

checkpoints = sorted(OUTPUT_ROOT.glob("checkpoint-epoch-*"))
assert checkpoints, "Training did not produce a checkpoint"
checkpoint = checkpoints[-1]
evaluation = json.loads((BUNDLE_ROOT / "eval/prompts.json").read_text())
EVAL_ROOT = OUTPUT_ROOT / "evaluation"
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
gc.collect()
torch.cuda.empty_cache()
tts = Qwen3TTSModel.from_pretrained(
    str(checkpoint),
    device_map="cuda:0",
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    attn_implementation=ATTENTION_IMPLEMENTATION,
)
for prompt in evaluation["prompts"]:
    wavs, sample_rate = tts.generate_custom_voice(text=prompt["text"], speaker=SPEAKER_NAME)
    sf.write(EVAL_ROOT / f"{prompt['id']}.wav", wavs[0], sample_rate)
print(sorted(path.name for path in EVAL_ROOT.glob("*.wav")))

In [ ]:
# Preserve the checkpoint privately in Google Drive.
# Large model files should not use browser download.
import datetime
import shutil

from google.colab import drive

drive.mount("/content/drive")
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
drive_target = Path("/content/drive/MyDrive/ChatWaifu-Nene-Qwen3-TTS") / stamp
drive_target.mkdir(parents=True, exist_ok=False)
shutil.copytree(checkpoint, drive_target / checkpoint.name)
shutil.copytree(EVAL_ROOT, drive_target / "evaluation")
shutil.copy2(OUTPUT_ROOT / "training_run.json", drive_target / "training_run.json")
shutil.copy2(BUNDLE_ROOT / "bundle_manifest.json", drive_target / "bundle_manifest.json")
print(drive_target)